In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog", "consumer_goods", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
base_path = f'/Volumes/consumer_goods/files/input_data_files/child_company/{data_source}'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"

print(catalog)
print(data_source)
print(base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

#pre-defining tables 
bronze_table = f"{catalog}.bronze.{data_source}"
silver_table = f"{catalog}.silver.{data_source}"
gold_table = f"{catalog}.gold.sb_fact_{data_source}"


consumer_goods
orders
/Volumes/consumer_goods/files/input_data_files/child_company/orders
Landing Path:  /Volumes/consumer_goods/files/input_data_files/child_company/orders/landing/
Processed Path:  /Volumes/consumer_goods/files/input_data_files/child_company/orders/processed/


In [0]:
df = (spark.read.format("csv")
        .option("header", "True")
        .option("inferSchema", "True")
        .load(f"{landing_path}/*.csv")
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*","_metadata.file_name")
)
print("Total Rows: ", df.count())
df.show(10)

Total Rows:  51810
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FOCT62720602|Tuesday, Septembe...|     ABC987|  25891301|     71.0|2026-03-03 12:09:...|orders_2025_09_30...|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891502|    125.0|2026-03-03 12:09:...|orders_2025_09_30...|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891403|    462.0|2026-03-03 12:09:...|orders_2025_09_30...|
|FOCT62720602|Tuesday, Septembe...|    INVALID|  25891601|    133.0|2026-03-03 12:09:...|orders_2025_09_30...|
|FOCT62720602|Tuesday, Septembe...|     789720|  25891602|     79.0|2026-03-03 12:09:...|orders_2025_09_30...|
|FOCT62720602|          2025/09/30|     XYZ123|  25891101|    472.0|2026-03-03 12:09:...|orde

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(bronze_table)

Moving files once read

In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(file_info.path,f"{processed_path}/{file_info.name}",True)

Keeping records where quantity is present

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
df_orders.show(2)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-03-03 12:09:...|orders_2025_07_01...|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
only showing top 2 rows


In [0]:
df_orders = df_orders.filter(F.col("order_qty").isNotNull())
df_orders.show(10)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891201|    354.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891501|    249.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          01-07-2025|     789320|  25891301|     46.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33401603|Tuesday, July 01,...|     789401|  25891302|     40.0|2026-03-03 12:09:...|orders_2025_07_01...|
|

checking for non-numeric customer-id and formatting date

In [0]:
df_orders = df_orders.withColumn("customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string"))
df_orders.show(10)    

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891201|    354.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891501|    249.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          01-07-2025|     789320|  25891301|     46.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33401603|Tuesday, July 01,...|     789401|  25891302|     40.0|2026-03-03 12:09:...|orders_2025_07_01...|
|

remove weekday in date

In [0]:
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

#Format date after

df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)
df_orders.show(10)


+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FJUL33320501|          2025-07-01|     789320|  25891203|    150.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          2025-07-01|     789320|  25891301|     46.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          2025-07-01|     789320|  25891201|    354.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          2025-07-01|     789320|  25891501|    249.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          2025-07-01|     789320|  25891301|     46.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33401603|          2025-07-01|     789401|  25891302|     40.0|2026-03-03 12:09:...|orders_2025_07_01...|
|

Duplicates

In [0]:
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

#cast product id
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))
df_orders.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+
|FJUL33320501|          2025-07-01|     789320|  25891203|    150.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          2025-07-01|     789320|  25891301|     46.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          2025-07-01|     789320|  25891201|    354.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33320501|          2025-07-01|     789320|  25891501|    249.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33401603|          2025-07-01|     789401|  25891302|     40.0|2026-03-03 12:09:...|orders_2025_07_01...|
|FJUL33401603|          2025-07-01|     789401|  25891502|    133.0|2026-03-03 12:09:...|orders_2025_07_01...|
|

Product code from products

In [0]:
df_products = spark.table("consumer_goods.silver.products")
df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

df_joined.show(5)

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+--------------------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|        product_code|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+--------------------+
|FJUL32202603|          2025-07-01|     789202|  25891103|    370.0|2026-03-03 12:09:...|orders_2025_07_01...|102628255d24304d6...|
|FJUL33321602|          2025-07-01|     789321|  25891602|     74.0|2026-03-03 12:09:...|orders_2025_07_01...|778c2a7aa27bfdb21...|
|FJUL34303602|          2025-07-01|     789303|  25891102|    317.0|2026-03-03 12:09:...|orders_2025_07_01...|e92c739a8d78cd6cb...|
|FJUL34622602|          2025-07-01|     789622|  25891402|    439.0|2026-03-03 12:09:...|orders_2025_07_01...|fe5a8036be4b9a787...|
|FJUL33402303|          2025-07-01|     789402|  25891303|     63.0|2026-03-

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

df_gold.show(2)

+------------+----------+-------------+--------------------+----------+-------------+
|    order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+------------+----------+-------------+--------------------+----------+-------------+
|FJUL32202603|2025-07-01|       789202|102628255d24304d6...|  25891103|        370.0|
|FJUL33321602|2025-07-01|       789321|778c2a7aa27bfdb21...|  25891602|         74.0|
+------------+----------+-------------+--------------------+----------+-------------+
only showing top 2 rows


In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

creating New Table


Merging with parent table

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-07-01|102628255d24304d6...|       789202|        370.0|
|2025-07-01|778c2a7aa27bfdb21...|       789321|         74.0|
|2025-07-01|e92c739a8d78cd6cb...|       789303|        317.0|
|2025-07-01|fe5a8036be4b9a787...|       789622|        439.0|
|2025-07-01|c68834ceaff15846b...|       789402|         63.0|
|2025-07-01|778c2a7aa27bfdb21...|       999999|        148.0|
|2025-07-01|778c2a7aa27bfdb21...|       789420|         76.0|
|2025-07-01|778c2a7aa27bfdb21...|       789201|         52.0|
|2025-07-01|e91ba9d665f90254d...|       789403|        379.0|
|2025-07-01|889c67757ece9c973...|       789421|         99.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


In [0]:
df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-07-01|102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268|789202       |2944.0       |
|2025-07-01|778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6|789321       |1742.0       |
|2025-07-01|e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e|789303       |4929.0       |
|2025-07-01|fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49|789622       |3810.0       |
|2025-07-01|c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f|789402       |960.0        |
+----------+----------------------------------------------------------------+-------------+-------------+
only showing top 5 rows


In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.gold.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
DESCRIBE HISTORY consumer_goods.gold.fact_orders

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-03-03T12:33:47.000Z,71320613334569,yvishnu224@gmail.com,MERGE,"Map(predicate -> [""(((date#18625 = date#18619) AND (product_code#18626 = product_code#18611)) AND (cast(customer_code#18627 as bigint) = cast(customer_code#18610 as bigint)))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1469607878050112),54744384-08fc-4bac-a81d-d3f9c32e17ad,0303-112818-wxc4dad7-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 14048, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2545, materializeSourceTimeMs -> 359, numTargetRowsInserted -> 3060, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1279, numTargetRowsUpdated -> 0, numOutputRows -> 3060, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3060, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 863)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-02T11:42:09.000Z,71320613334569,yvishnu224@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(903424024329876),1ae2dd3f-8701-4ff9-aa7d-a967e6342af2,0302-113816-lp5wn68f-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 132760, numDeletionVectorsRemoved -> 0, numOutputRows -> 93055, numOutputBytes -> 132760)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-02T11:39:56.000Z,71320613334569,yvishnu224@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(903424024329876),c6fb62cd-90ab-4010-9ec2-2d0817eff2f2,0302-113816-lp5wn68f-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 93055, numOutputBytes -> 132760)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
